# Mixed-effects pipeline (primary study model)
This notebook builds the final mixed-effects pipeline for the study, aligned with psycholinguistics practice:
- Outcome: log total reading time (TRT) with light trimming.
- Predictors: standardized frequency, length, and locality metrics; sum-coded condition.
- Random effects: crossed random intercepts (subject + item); optional subject slopes for frequency and length if stable/improving fit.
- Exports: fixed effects table, model choice/metrics, residual diagnostics, and simple fit summary.

In [3]:
# Imports and data load
from pathlib import Path
import json
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm
import matplotlib.pyplot as plt

BASE = Path('data-clean') / 'processed'
PARQ = BASE / 'trt_with_features.parquet'
CSV = BASE / 'trt_with_features.csv'
OUT = BASE
OUT.mkdir(parents=True, exist_ok=True)

df = pl.read_parquet(PARQ) if PARQ.exists() else pl.read_csv(CSV)
use = (df
       .select(['total_reading_time','subject_id','stimulus','condition',
                'word_len','freq_zipf','dep_dist','depth','integration_cost'])
       .drop_nulls(subset=['total_reading_time','word_len','freq_zipf'])
).to_pandas()

# Ensure identifiers are strings (safer for formulas)
use['subject_id'] = use['subject_id'].astype(str)
use['stimulus'] = use['stimulus'].astype(str)

# Light trimming and log-transform (literature-aligned)
use = use[(use['total_reading_time'] >= 150) & (use['total_reading_time'] <= 4000)].copy()
use['log_trt'] = np.log(use['total_reading_time'])

# Standardize continuous predictors
for c in ['word_len','freq_zipf','dep_dist','depth','integration_cost']:
    s = use[c].std()
    use[c + '_z'] = (use[c] - use[c].mean()) / (s if s and not np.isnan(s) else 1.0)

print('Rows after trimming:', len(use))
use.head(3)

Rows after trimming: 18172


,total_reading_time,subject_id,stimulus,condition,word_len,freq_zipf,dep_dist,depth,integration_cost,log_trt,word_len_z,freq_zipf_z,dep_dist_z,depth_z,integration_cost_z
1,686,P01,voicemail-neg.naturalness,neg,7,4.626853,3,1,1.5,6.530878,0.741381,-0.561708,0.210652,-0.771338,0.215781
18,411,P01,voicemail-neg.naturalness,neg,6,5.057742,1,3,0.6,6.018593,0.328054,-0.247108,-0.470300,0.550312,-0.394412
21,266,P01,voicemail-neg.naturalness,neg,4,6.093859,1,3,0.6,5.583496,-0.498598,0.509382,-0.470300,0.550312,-0.394412


In [4]:
# Fit primary model: crossed random intercepts (subject + item)
vc = {'stimulus': '0 + C(stimulus)'}
formula = 'log_trt ~ freq_zipf_z + word_len_z + dep_dist_z + depth_z + integration_cost_z + C(condition, Sum)'
m_crossed = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1', data=use)
res_crossed = m_crossed.fit(method='lbfgs', reml=True)
print('=== MixedLM (crossed intercepts) ===')
print(res_crossed.summary())

# Attempt subject slopes (freq & length) + item VC; fall back on failure
res_slopes = None
try:
    m_slopes = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1 + freq_zipf_z + word_len_z', data=use)
    res_slopes = m_slopes.fit(method='lbfgs', reml=True)
    print('=== MixedLM (subject slopes: freq, length) + item VC ===')
    print(res_slopes.summary())
except Exception as e:
    print('[Info] Subject-slopes model failed:', repr(e))

# Choose model (use ML AIC if available; otherwise slope-variance heuristic)
chosen = 'crossed'
chosen_res = res_crossed

# Try ML AIC selection without altering REML reporting
aic_ml = {'crossed': None, 'slopes': None}
try:
    m_crossed_ml = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1', data=use)
    res_crossed_ml = m_crossed_ml.fit(method='lbfgs', reml=False)
    aic_ml['crossed'] = float(getattr(res_crossed_ml, 'aic', np.nan))
    if res_slopes is not None:
        m_slopes_ml = sm.MixedLM.from_formula(formula, groups='subject_id', vc_formula=vc, re_formula='1 + freq_zipf_z + word_len_z', data=use)
        res_slopes_ml = m_slopes_ml.fit(method='lbfgs', reml=False)
        aic_ml['slopes'] = float(getattr(res_slopes_ml, 'aic', np.nan))
except Exception as e:
    print('[Info] ML AIC selection skipped:', repr(e))

prefer = False
if res_slopes is not None:
    # Prefer slopes if ML AIC improves
    if aic_ml['crossed'] is not None and aic_ml['slopes'] is not None and np.isfinite(aic_ml['crossed']) and np.isfinite(aic_ml['slopes']):
        prefer = aic_ml['slopes'] < aic_ml['crossed']
    # Or if slope variances are clearly > 0
    try:
        diag = np.diag(res_slopes.cov_re)
        slope_variance_positive = (len(diag) >= 3) and ((float(diag[1]) > 1e-4) or (float(diag[2]) > 1e-4))
        prefer = prefer or slope_variance_positive
    except Exception:
        pass
    if prefer:
        chosen = 'slopes'
        chosen_res = res_slopes

print(f'Chosen model: {chosen}')

# Export fixed effects and decision
fe = pd.DataFrame({'coef': chosen_res.params, 'se': chosen_res.bse})
fe.to_csv(OUT / 'mixedlm_fixed_effects_chosen.csv')
choice = {
    'chosen': chosen,
    'aic': {
        'crossed': float(getattr(res_crossed, 'aic', np.nan)),
        'slopes': float(getattr(res_slopes, 'aic', np.nan)) if res_slopes is not None else None
    },
    'aic_ml': aic_ml,
    'llf': {
        'crossed': float(res_crossed.llf),
        'slopes': float(res_slopes.llf) if res_slopes is not None else None
    }
}
with open(OUT / 'mixedlm_choice.json', 'w', encoding='utf-8') as f:
    json.dump(choice, f, indent=2)
print('Saved: mixedlm_fixed_effects_chosen.csv, mixedlm_choice.json')

# Residual diagnostics: fitted vs observed, residual histogram
try:
    fitted = chosen_res.fittedvalues
    resid = use.loc[fitted.index, 'log_trt'] - fitted
    # Scatter plot
    plt.figure(figsize=(5,4))
    plt.scatter(fitted, use.loc[fitted.index, 'log_trt'], s=4, alpha=0.3)
    plt.xlabel('Fitted log(TRT)')
    plt.ylabel('Observed log(TRT)')
    plt.title('Observed vs Fitted (log scale)')
    plt.tight_layout()
    plt.savefig(OUT / 'mixedlm_obs_vs_fitted.png', dpi=150)
    plt.close()
    # Histogram of residuals
    plt.figure(figsize=(5,4))
    plt.hist(resid, bins=50, alpha=0.8)
    plt.xlabel('Residual (log TRTs)')
    plt.ylabel('Count')
    plt.title('Residuals histogram')
    plt.tight_layout()
    plt.savefig(OUT / 'mixedlm_residuals_hist.png', dpi=150)
    plt.close()
    print('Saved: mixedlm_obs_vs_fitted.png, mixedlm_residuals_hist.png')
except Exception as e:
    print('[Info] Diagnostics plotting skipped:', repr(e))

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


=== MixedLM (crossed intercepts) ===
               Mixed Linear Model Regression Results
Model:                 MixedLM    Dependent Variable:    log_trt    
No. Observations:      18172      Method:                REML       
No. Groups:            12         Scale:                 0.1885     
Min. group size:       902        Log-Likelihood:        -11022.9610
Max. group size:       2001       Converged:             Yes        
Mean group size:       1514.3                                       
--------------------------------------------------------------------
                         Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------
Intercept                 5.724    0.027 209.251 0.000  5.670  5.777
C(condition, Sum)[S.neg] -0.011    0.011  -0.979 0.327 -0.032  0.011
C(condition, Sum)[S.pos]  0.024    0.011   2.182 0.029  0.002  0.045
freq_zipf_z              -0.048    0.005  -9.699 0.000 -0.057 -0.038
word_len_z   

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


=== MixedLM (subject slopes: freq, length) + item VC ===
                 Mixed Linear Model Regression Results
Model:                 MixedLM      Dependent Variable:      log_trt    
No. Observations:      18172        Method:                  REML       
No. Groups:            12           Scale:                   0.1860     
Min. group size:       902          Log-Likelihood:          -10921.4339
Max. group size:       2001         Converged:               Yes        
Mean group size:       1514.3                                           
------------------------------------------------------------------------
                             Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------------------
Intercept                     5.729    0.029 200.058 0.000  5.672  5.785
C(condition, Sum)[S.neg]     -0.014    0.011  -1.257 0.209 -0.035  0.008
C(condition, Sum)[S.pos]      0.028    0.011   2.583 0.010  0.007  0.050
freq_zipf_z 

C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
C:\Python312\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Chosen model: slopes
Saved: mixedlm_fixed_effects_chosen.csv, mixedlm_choice.json
Saved: mixedlm_obs_vs_fitted.png, mixedlm_residuals_hist.png
